# 03 — Context dependence

**The core notebook.** Which perturbations behave differently depending on
immune environment?

Two independent readouts, deliberately not one:

1. **Signature divergence** — correlate each perturbation's log2FC signature in
   IFN-γ and co-culture against its signature in control.
2. **E-distance** (`pertpy`, permutation null) — a model-free answer to "did
   this perturbation do anything at all in this condition."

A perturbation is called context-dependent only if **both** hold: a real
effect somewhere, and divergence between environments. Requiring only
divergence would fill the hit list with noise, because two null signatures are
also uncorrelated. That conjunction is the analytical core of the project.

In [ ]:
# =============================================================================
# nb03 — Embedding and structure
#
# Not a clustering notebook. The dominant axis of variation here is condition,
# so clustering cells would recover the experimental design and invite the
# circularity of testing on groups defined by the same expression data.
#
# What the embedding is for: confirming the design held (do the three arms
# separate?), finding structure that was NOT designed and therefore acts as a
# confounder (cell cycle, depth), and checking for T-cell contamination in the
# co-culture arm.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
GUIDE      = s["guide"]
CTRL       = cfg["schema"]["control_label"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]


cc = load_panels()["cell_cycle"]
s_genes   = [g for g in cc["s_phase"]   if g in rna_n.var_names]
g2m_genes = [g for g in cc["g2m_phase"] if g in rna_n.var_names]
print(f"S: {len(s_genes)}/{len(cc['s_phase'])}, "
      f"G2M: {len(g2m_genes)}/{len(cc['g2m_phase'])}")

sc.tl.score_genes_cell_cycle(rna_n, s_genes=s_genes, g2m_genes=g2m_genes)
print(rna_n.obs["phase"].value_counts())


# QC flags from nb02, for annotating anything that turns up here
flags = pd.read_csv(P.tables / "02_gene_flags.csv", index_col=0)
elig  = pd.read_csv(P.tables / "02_guide_eligibility.csv", index_col=0)

print(f"RNA: {rna.n_obs:,} cells x {rna.n_vars:,} genes")
print(f"ADT: {adt.n_obs:,} cells x {adt.n_vars:,} features")
print(f"authors' UMAP present: {'X_umap_orig' in rna.obsm}")
assert set(rna.obs['MOI'].unique()) == {1}

In [ ]:
# Work on a copy — the .h5mu on disk keeps raw counts, and nb04's DE run
# needs those untouched.
# ---- normalise ------------------------------------------------------------
rna_n = rna.copy()
rna_n.X = rna_n.layers["counts"].copy()

sc.pp.normalize_total(rna_n, target_sum=1e4)
sc.pp.log1p(rna_n)

# ---- cell cycle scoring ---------------------------------------------------
# Must happen HERE, on the full gene set. Most cell-cycle genes are not
# variable in control cells and so will not survive HVG selection — scoring
# after the subset would use a fraction of each gene set and give unreliable
# scores. The resulting columns live in .obs and survive the subset.
cc = panels["cell_cycle"]
s_genes   = [g for g in cc["s_phase"]   if g in rna_n.var_names]
g2m_genes = [g for g in cc["g2m_phase"] if g in rna_n.var_names]
print(f"cell cycle genes found — S: {len(s_genes)}/{len(cc['s_phase'])}, "
      f"G2M: {len(g2m_genes)}/{len(cc['g2m_phase'])}")

sc.tl.score_genes_cell_cycle(rna_n, s_genes=s_genes, g2m_genes=g2m_genes)
print(rna_n.obs["phase"].value_counts())

# ---- HVG on control cells only --------------------------------------------
ctrl_cells = rna_n.obs[PERT].astype(str) == CTRL
print(f"control cells: {ctrl_cells.sum():,}")

ctrl_sub = rna_n[ctrl_cells].copy()
sc.pp.highly_variable_genes(ctrl_sub, n_top_genes=cfg["de"]["n_hvg"],
                            flavor="seurat_v3", layer="counts")
rna_n.var["highly_variable"] = rna_n.var_names.isin(
    ctrl_sub.var_names[ctrl_sub.var["highly_variable"]]
)
print(f"HVGs: {rna_n.var['highly_variable'].sum()}")

# ---- scale, PCA, neighbours, UMAP -----------------------------------------
rna_n = rna_n[:, rna_n.var["highly_variable"]].copy()
sc.pp.scale(rna_n, max_value=10)
sc.tl.pca(rna_n, n_comps=50, svd_solver="arpack", random_state=SEED)


In [ ]:
# ---- how many PCs carry signal? -------------------------------------------
sc.pl.pca_variance_ratio(rna_n, n_pcs=50, log=True)

# What loads on the first few PCs. If PC1 is GNLY / CCL5 / NKG7, the dominant
# axis is T-cell contamination rather than biology, and those cells should be
# removed before the embedding means anything.
sc.pl.pca_loadings(rna_n, components=[1, 2, 3])

In [ ]:
# ---- neighbours + UMAP ----------------------------------------------------
N_PCS = 22                       # set from the elbow plot above

sc.pp.neighbors(rna_n, n_neighbors=15, n_pcs=N_PCS, random_state=SEED)
sc.tl.umap(rna_n, random_state=SEED)

# carry the authors' coordinates across for the comparison panel
if "X_umap_orig" in rna.obsm:
    rna_n.obsm["X_umap_orig"] = rna.obsm["X_umap_orig"][
        rna.obs_names.get_indexer(rna_n.obs_names)
    ]

print(rna_n.obsm.keys())

In [ ]:
# ---- quick look -----------------------------------------------------------
# Contamination markers first — this determines whether the embedding is
# usable as-is or needs a cell-removal step.
tcell = [g for g in ["PTPRC", "CD3D", "CD3E", "CD2", "GNLY", "NKG7",
                     "GZMB", "CCL5", "IL32"] if g in rna_n.var_names]
print(f"T-cell markers present in HVG set: {tcell}")

sc.pl.umap(rna_n, color=[COND] + tcell, ncols=3,
           palette=[pal[c] for c in cond_order], frameon=False, s=3)

In [ ]:
# ---- pick a resolution ----------------------------------------------------
# Target is ~3-4 subclusters per condition plus the contaminating T-cell
# island, so roughly 10-13 clusters. Resolution 1.0 gave 27 — too fine for a
# figure, though it was useful for isolating small populations.
# Choose from the composition table, not from how the UMAP looks.
for res in [0.2, 0.3, 0.4, 0.5]:
    key = f"leiden_{res}"
    sc.tl.leiden(rna_n, resolution=res, key_added=key, random_state=SEED,
                 flavor="igraph", n_iterations=2, directed=False)
    n = rna_n.obs[key].nunique()
    # does the T-cell island survive as its own cluster at this resolution?
    tc = rna_n.obs.loc[rna_n.obs["leiden"] == "7", key].value_counts()
    print(f"res {res}: {n:2d} clusters | old cluster 7 lands in "
          f"{tc.index[0]} ({tc.iloc[0]/tc.sum():.0%} of it)")

In [ ]:
RES = 0.3                                    # set from the sweep above
rna_n.obs["cluster"] = rna_n.obs[f"leiden_{RES}"]

print(rna_n.obs["cluster"].value_counts().sort_index())
print()
print(pd.crosstab(rna_n.obs["cluster"], rna_n.obs[COND], normalize="index").round(2))

# flag the T-cell cluster by marker expression rather than by eye
tcell = [g for g in ["PTPRC","CD3D","CD3E","CD2","GNLY","NKG7","GZMB","CCL5","IL32"]
         if g in rna_n.var_names]
tc_score = pd.DataFrame(
    rna_n[:, tcell].X.toarray() if sp.issparse(rna_n.X) else rna_n[:, tcell].X,
    index=rna_n.obs_names, columns=tcell
).mean(axis=1).groupby(rna_n.obs["cluster"]).mean()
print("\nmean T-cell marker score per cluster:")
print(tc_score.sort_values(ascending=False).round(2))

In [ ]:
# ---- which perturbations are non-randomly distributed across clusters? -----
# Confound: clusters are almost perfectly nested within conditions, so raw
# cluster enrichment would mostly recover condition composition. Computed
# WITHIN each condition instead, so the question becomes: given a cell is in
# the IFN-γ arm, does its perturbation predict which IFN-γ subcluster it lands
# in? That is a real question about perturbation-driven state.
MIN_CELLS = 30

rows = []
for cond in cond_order:
    sub = rna_n.obs[rna_n.obs[COND].astype(str) == cond]
    cl_frac = sub["cluster"].value_counts(normalize=True)      # expected
    for pert, d in sub.groupby(PERT, observed=True):
        if len(d) < MIN_CELLS or str(pert) == CTRL:
            continue
        obs = d["cluster"].value_counts(normalize=True)
        for cl in cl_frac.index:
            if cl_frac[cl] < 0.02:            # ignore tiny clusters
                continue
            rows.append({
                "perturbation": str(pert), "condition": cond, "cluster": cl,
                "n": len(d),
                "log2_enrich": np.log2((obs.get(cl, 0) + 0.01) / (cl_frac[cl] + 0.01)),
            })

enrich = pd.DataFrame(rows)
top = (enrich.assign(a=enrich["log2_enrich"].abs())
       .groupby("perturbation")["a"].max()
       .sort_values(ascending=False))
print(top.head(20).round(2))

TOP_N = 12
top_perts = top.head(TOP_N).index.tolist()

In [ ]:
# =============================== FIGURE 3 ==================================
# Three-panel UMAP: what the clusters are, what drives them, and whether any
# perturbation is strong enough to move cells against that dominant axis.
fig, ax = plt.subplot_mosaic(
    """
    ABC
    """,
    figsize=(19, 6.5),
)

emb = rna_n.obsm["X_umap"]

# ---------------------------------- A --------------------------------------
# Leiden clusters. Used to DESCRIBE structure and isolate the contaminating
# population — not as groups for differential testing, which would be circular
# given they were defined from the same expression data.
clusters = rna_n.obs["cluster"].astype(str)
order = sorted(clusters.unique(), key=int)
cmap = plt.cm.tab20(np.linspace(0, 1, len(order)))

for i, cl in enumerate(order):
    m = (clusters == cl).values
    ax["A"].scatter(emb[m, 0], emb[m, 1], s=1.2, alpha=0.5,
                    color=cmap[i], linewidths=0, rasterized=True)
    # median, not mean — robust to stragglers pulling the label off-cluster
    ax["A"].text(np.median(emb[m, 0]), np.median(emb[m, 1]), cl,
                 fontsize=9, fontweight="bold", ha="center", va="center",
                 bbox=dict(boxstyle="circle,pad=0.2", facecolor="white",
                           edgecolor="none", alpha=0.75))

ax["A"].text(0.01, 0.99,
             f"{len(order)} clusters, res = {RES}\n"
             f"{rna_n.n_obs:,} cells\n"
             f"cluster 12 = T cells (n=577)",
             transform=ax["A"].transAxes, ha="left", va="top",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))

# ---------------------------------- B --------------------------------------
# Condition. Expected to recapitulate panel A almost exactly — the treatment
# is the dominant axis of variation, which is why clusters are not used as
# testing groups.
for cond in cond_order:
    m = (rna_n.obs[COND].astype(str) == cond).values
    ax["B"].scatter(emb[m, 0], emb[m, 1], s=1.2, alpha=0.5,
                    color=pal.get(cond, "#888"), linewidths=0,
                    label=f"{cond} (n={m.sum():,})", rasterized=True)

ax["B"].legend(fontsize=9, loc="upper left", markerscale=8, framealpha=0.9)

# ---------------------------------- C --------------------------------------
# Perturbations whose cells are most unevenly distributed across subclusters
# WITHIN their own condition. Computed within condition because clusters are
# nearly synonymous with treatment — a raw enrichment would mostly rank
# perturbations by which arm they happen to be abundant in.
ax["C"].scatter(emb[:, 0], emb[:, 1], s=0.8, color="#dddddd",
                linewidths=0, rasterized=True)

dcmap = plt.cm.tab20(np.linspace(0, 1, len(top_perts)))
pert_col = rna_n.obs[PERT].astype(str).values
for i, p in enumerate(top_perts):
    m = pert_col == p
    ax["C"].scatter(emb[m, 0], emb[m, 1], s=3.5, alpha=0.85,
                    color=dcmap[i], linewidths=0,
                    label=f"{p} ({m.sum():,})", rasterized=True)

ax["C"].legend(fontsize=6.5, loc="upper left", markerscale=3.5, ncol=2,
               framealpha=0.9)

# ---- shared axis styling --------------------------------------------------
for k in ["A", "B", "C"]:
    ax[k].set_xticks([]); ax[k].set_yticks([])
    ax[k].set_xlabel("UMAP 1"); ax[k].set_ylabel("UMAP 2")

titles = {
    "A": f"Leiden clusters (res {RES})",
    "B": "condition",
    "C": f"top {TOP_N} perturbations by cluster enrichment",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

fig.tight_layout()
savefig(fig, "03_embedding", cfg)

In [ ]:
# =============================== FIGURE 4 ==================================
# Cluster identity, two ways.
# A) Markers derived from the data — what actually distinguishes each cluster.
# B) Hand-picked programs — do known biology sets land where expected?
#
# Caveat on A: clusters were defined from expression, so testing genes against
# them is circular and the p-values are not valid inference. Used to LABEL
# clusters, not to discover anything. It earns its place because the genes it
# surfaces are unconstrained by prior expectation, which B is not.

# ---- data-derived markers -------------------------------------------------
sc.tl.rank_genes_groups(rna_n, groupby="cluster", method="wilcoxon",
                        key_added="cluster_markers")

names = pd.DataFrame(rna_n.uns["cluster_markers"]["names"])
seen, derived = set(), []
for cl in rna_n.obs["cluster"].cat.categories:
    for g in names[cl].head(3):
        if g not in seen:
            seen.add(g); derived.append(g)

# ---- hand-picked programs -------------------------------------------------
marker_sets = {
    "T cell":  ["PTPRC", "CD3D", "CD2", "GNLY", "NKG7", "CCL5"],
    "IFN-γ":   ["STAT1", "IRF1", "GBP1", "GBP2", "CXCL10", "SOCS1"],
    "MHC-I":   ["B2M", "HLA-A", "HLA-B", "TAP1", "PSMB9"],
    "inflam.": ["IL1B", "CXCL8", "CCL2", "MMP1", "S100A8"],
    "cycle":   ["MKI67", "TOP2A", "CCNB2"],
}
marker_sets = {k: [g for g in v if g in rna_n.var_names] for k, v in marker_sets.items()}
marker_sets = {k: v for k, v in marker_sets.items() if v}
curated = [g for gs in marker_sets.values() for g in gs]

order = sorted(rna_n.obs["cluster"].astype(str).unique(), key=int)
grp = rna_n.obs["cluster"].astype(str).values


def dotplot_panel(a, genes, sets=None):
    """Hand-built dotplot: size = fraction expressing, colour = scaled mean.

    Built manually because sc.pl.dotplot manages its own figure and will not
    draw into a mosaic axes.
    """
    X = rna_n[:, genes].X
    X = np.asarray(X.todense()) if sp.issparse(X) else np.asarray(X)
    Xdf = pd.DataFrame(X, index=rna_n.obs_names, columns=genes)

    frac = Xdf.gt(0).groupby(grp).mean().loc[order]
    mean = Xdf.groupby(grp).mean()
    mean = ((mean - mean.min()) / (mean.max() - mean.min())).loc[order]

    gx, gy = np.meshgrid(np.arange(len(genes)), np.arange(len(order)))
    h = a.scatter(gx.ravel(), gy.ravel(), s=frac.values.ravel() * 110,
                  c=mean.values.ravel(), cmap="Blues",
                  edgecolor="grey", linewidth=0.2, vmin=0, vmax=1)

    a.set_xticks(range(len(genes)))
    a.set_xticklabels(genes, rotation=90, fontsize=6.5)
    a.set_yticks(range(len(order)))
    a.set_yticklabels(order, fontsize=8)
    a.set_ylabel("cluster")
    a.set_xlim(-0.7, len(genes) - 0.3)
    # reversed limits instead of invert_yaxis(), leaving headroom for brackets
    a.set_ylim(len(order) - 0.4, -2.2 if sets else -0.7)
    a.grid(alpha=0.15)

    if sets:
        b = 0
        for name, gs in sets.items():
            n = len(gs)
            a.axvline(b - 0.5, color="k", lw=0.6)
            a.text(b + n / 2 - 0.5, -1.5, name, ha="center",
                   fontsize=7.5, fontweight="bold")
            b += n
    return h


fig, ax = plt.subplot_mosaic(
    """
    AAAA
    BBBB
    """,
    figsize=(17, 12),
)

h1 = dotplot_panel(ax["A"], derived)
h2 = dotplot_panel(ax["B"], curated, sets=marker_sets)
plt.colorbar(h1, ax=ax["A"], label="scaled mean expr.", shrink=0.7)
plt.colorbar(h2, ax=ax["B"], label="scaled mean expr.", shrink=0.7)

titles = {
    "A": f"data-derived markers (top 3 per cluster, {len(derived)} unique)",
    "B": "curated marker programs",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=10)

fig.tight_layout()
savefig(fig, "03_cluster_markers", cfg)

In [ ]:
# =============================== FIGURE 5 ==================================
# QC and cell-cycle overlays on the embedding.
#
# The question these answer is not "is the data good" — that was settled in
# nb01. It is whether any region of the embedding is driven by TECHNICAL
# variation rather than biology, and whether cell cycle pulls cells out of
# their condition territory strongly enough to confound perturbation effects.
#
# Worth reading against the nb01 violins: those showed co-culture has higher
# mito on AVERAGE. These show whether that is a discrete subpopulation or a
# global shift — which are different, and matter differently for nb05.

emb = rna_n.obsm["X_umap"]

overlays = [
    ("ncounts",      "UMIs per cell",        "viridis",  True),   # (col, label, cmap, log)
    ("ngenes",       "genes per cell",       "viridis",  True),
    ("percent_mito", "% mitochondrial",      "magma",    False),
    ("percent_ribo", "% ribosomal",          "magma",    False),
    ("S_score",      "S-phase score",        "RdBu_r",   False),
    ("G2M_score",    "G2/M score",           "RdBu_r",   False),
]
overlays = [o for o in overlays if o[0] in rna_n.obs.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 9.5))
axes = axes.ravel()

for a, (col, label, cm, logscale) in zip(axes, overlays):
    v = rna_n.obs[col].values.astype(float)
    if logscale:
        v = np.log10(v + 1)
        label = f"log10({label} + 1)"

    # clip to 1st-99th percentile so a handful of extreme cells do not
    # compress the entire colour range
    lo, hi = np.percentile(v, [1, 99])

    # plot in ascending order so high-value cells are drawn on top rather than
    # being buried under the majority
    o = np.argsort(v)
    h = a.scatter(emb[o, 0], emb[o, 1], c=v[o], s=1.2, alpha=0.6,
                  cmap=cm, vmin=lo, vmax=hi, linewidths=0, rasterized=True)
    plt.colorbar(h, ax=a, shrink=0.75, label=label)
    a.set_title(label, fontsize=10)

# ---- discrete overlay: cell-cycle phase -----------------------------------
a = axes[len(overlays)]
phase_cols = {"G1": "#c9c9c9", "S": "#1F6FB2", "G2M": "#C2185B"}
for ph, c in phase_cols.items():
    m = (rna_n.obs["phase"].astype(str) == ph).values
    if m.sum() == 0:
        continue
    a.scatter(emb[m, 0], emb[m, 1], s=1.2, alpha=0.55, color=c,
              linewidths=0, label=f"{ph} ({m.sum()/len(m):.0%})",
              rasterized=True)
a.legend(fontsize=8, loc="upper left", markerscale=8, framealpha=0.9)
a.set_title("cell-cycle phase", fontsize=10)

# ---- authors' embedding, for comparison -----------------------------------
a = axes[len(overlays) + 1]
if "X_umap_orig" in rna_n.obsm:
    o_emb = rna_n.obsm["X_umap_orig"]
    for cond in cond_order:
        m = (rna_n.obs[COND].astype(str) == cond).values
        a.scatter(o_emb[m, 0], o_emb[m, 1], s=1.2, alpha=0.5,
                  color=pal.get(cond, "#888"), linewidths=0, rasterized=True)
    a.set_title("authors' UMAP (Frangieh et al.)", fontsize=10)
else:
    a.set_axis_off()

for a in axes:
    a.set_xticks([]); a.set_yticks([])

# panel letters
for i, a in enumerate(axes):
    if a.has_data():
        a.text(-0.02, 1.06, f"{chr(65 + i)})", transform=a.transAxes,
               fontweight="bold", fontsize=12, ha="left", va="top")
    else:
        a.set_axis_off()

fig.tight_layout()
savefig(fig, "03_umap_overlays", cfg)

In [ ]:
overlays = [
    ("ncounts",      "UMIs per cell",   "viridis", True),
    ("ngenes",       "genes per cell",  "viridis", True),
    ("percent_mito", "% mitochondrial", "magma",   False),
    ("percent_ribo", "% ribosomal",     "magma",   False),
]
overlays = [o for o in overlays if o[0] in rna_n.obs.columns]

fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for a, (col, label, cm, logscale) in zip(axes, overlays):
    v = rna_n.obs[col].values.astype(float)
    if logscale:
        v = np.log10(v + 1)
        label = f"log10({label} + 1)"

    # clip to 1st-99th percentile so a few extreme cells don't compress the range
    lo, hi = np.percentile(v, [1, 99])
    # draw in ascending order so high-value cells sit on top rather than being
    # buried under the majority — at 127k points, draw order is what you see
    o = np.argsort(v)
    h = a.scatter(emb[o, 0], emb[o, 1], c=v[o], s=1.2, alpha=0.6,
                  cmap=cm, vmin=lo, vmax=hi, linewidths=0, rasterized=True)
    plt.colorbar(h, ax=a, shrink=0.75, label=label)
    a.set_title(label, fontsize=10)

# cell-cycle phase — the discrete summary makes the continuous S/G2M scores
# redundant, so only this is shown
a = axes[len(overlays)]
for ph, c in {"G1": "#c9c9c9", "S": "#1F6FB2", "G2M": "#C2185B"}.items():
    m = (rna_n.obs["phase"].astype(str) == ph).values
    if m.sum():
        a.scatter(emb[m, 0], emb[m, 1], s=1.2, alpha=0.55, color=c,
                  linewidths=0, label=f"{ph} ({m.sum()/len(m):.0%})",
                  rasterized=True)
a.legend(fontsize=8, loc="upper left", markerscale=8, framealpha=0.9)
a.set_title("cell-cycle phase", fontsize=10)

for i, a in enumerate(axes):
    a.set_xticks([]); a.set_yticks([])
    a.text(-0.02, 1.08, f"{chr(65 + i)})", transform=a.transAxes,
           fontweight="bold", fontsize=12, ha="left", va="top")

fig.tight_layout()
savefig(fig, "03_umap_overlays", cfg)

In [ ]:
# add to the end of nb03
emb_out = rna_n.obs[["cluster", "phase", "S_score", "G2M_score"]].copy()
emb_out[["umap1", "umap2"]] = rna_n.obsm["X_umap"]
emb_out.to_parquet(P.data_interim / "03_embedding.parquet")